- 免密登录

```
ssh-keygen -t ed25519 -a 100 \
  -f ~/.ssh/id_ed25519_h100 \
  -C "$(whoami)@$(hostname)-h100"

ssh-add --apple-use-keychain ~/.ssh/id_ed25519_h100

ssh-copy-id -i ~/.ssh/id_ed25519_h100.pub xx@xx
ssh-copy-id -i ~/.ssh/id_ed25519_h100.pub xx@xx

vim ~/.ssh/config
```
- Supermicro 8U SYS-821GE-TNHR
    - https://www.supermicro.com/manuals/superserver/8U/MNL-2596.pdf
    - 第16页：主板托盘、CPU、DIMM 和 PCIe Switch Module 顶视图
    - 第18页：HGX H100 8-GPU Baseboard / GPU Tray 顶视图
    - 第19页：完整 CPU–PCIe Switch–NIC–NVMe–HGX 系统框图
    - 第23页：双路 CPU、DDR5、UPI、MCIO、PCH 主板框图

```
dmidecode -s system-manufacturer
dmidecode -s system-product-name
dmidecode -s baseboard-manufacturer
dmidecode -s baseboard-product-name

整机厂商：Supermicro
整机型号：SYS-821GE-TNHR
主板厂商：Supermicro
主板型号：X13DEG-OAD
```

### h100

- h200 > h100 > h800 > h20
- hbm
    - a100: hbm2e, 80GB; h100: hbm3, 80GB
    - H200(141GB, 1.76*80)

| GPU | 架构 | Compute Capability | CUDA SASS目标 |
|---|---|---:|---|
| NVIDIA A100 | Ampere | 8.0 | `sm_80` |
| RTX 4090 | Ada Lovelace | 8.9 | `sm_89` |
| NVIDIA H100 | Hopper | 9.0 | `sm_90` |

#### pcie vs. sxm5

- NVIDIA HGX H100 8-GPU Baseboard
- GPU-GPU 互连	4×第三代 NVSwitch + 第四代 NVLink
- CPU-GPU PCIe	每卡 PCIe 5.0 x16，实测 32 GT/s x16

### 节点间连接

- 双方免密登录
- 两台服务器还有八组一一对应的地址
    - 两机之间管理网 + 8 条 RoCE 网段（172.20.101–108.x）互 ping 全通
- RDMA 有三种落地方式
    - InfiniBand
    - RoCEv2: 标准以太网，RDMA 报文塞进 UDP/IP

```
# 通过 link layer 判断
> ibstat | grep -E "^CA |Link layer"
CA 'mlx5_XX'
		Link layer: Ethernet
> cat /sys/class/infiniband/mlx5_0/ports/1/link_layer
Ethernet
```

下面这些不论 IB 还是 RoCE 都长一个样，不能作为判据：

- 设备名 mlx5_0（mlx5 是驱动名，不是协议名）
- 环境变量 NCCL_IB_HCA、NCCL_IB_GID_INDEX（历史包袱，RoCE 也用这套）
- 工具名 ib_write_bw、ibstat、ibv_devinfo
- /dev/infiniband/*、内核模块 ib_core

### topo 与带宽

- nvidia-smi topo -m
    - 8× H100 SXM 80GB，任意两卡之间 NV18 = 18 条 NVLink4 走 NVSwitch 全互联
    - GPU0 有 18 条 NVLink4 接进交换网，通过交换网它能以全带宽到达任意一个 GPU。NV18 描述的是"这个 GPU 的接入带宽是 18 条 link"，不是点对点线缆数。18 条 link 是绑定聚合使用的（类似链路聚合），一次传输在 18 条上做条带化。
- 每链路 26.562 GB/s × 18 = 478 GB/s 单向 / 900 GB/s 双向，满配无缺链
    - nvidia-smi nvlink -s

> N 个 GPU，每个手上有一份长度 S 的数据（比如梯度）。目标：每个 GPU 最后都拿到这 N 份的逐元素求和。

- 朴素做法（全发给 rank0 求和再广播）会让 rank0 的网卡成为瓶颈。NCCL 用的是 ring all-reduce，拆成两个阶段：
    - 阶段一 reduce-scatter（N−1 步）：把 S 切成 N 块，环上转 N−1 圈，每步每个 GPU 发一块给下家、收一块加到自己身上。结束时每个 GPU 持有 1/N 的完整求和结果。
    - 阶段二 all-gather（N−1 步）：再转 N−1 圈，把各自那 1/N 传遍全环。
- 每个 GPU 总共发送：
    - $2\cdot(N-1)\cdot\frac{S}N$
    - 注意这个量与 N 几乎无关（N 大时趋近 2S）——这就是 ring all-reduce 的漂亮之处。
- 设每个 GPU 的出口链路带宽为 $B$，则耗时 $t = \dfrac{2(N-1)S/N}{B}$。
    - algbw（算法带宽）：$\dfrac{S}{t}$，用户视角：我提交了 S 字节的 allreduce，等效吞吐多少
    - busbw（总线带宽）：$\text{algbw} \times \dfrac{2(N-1)}{N}$，硬件视角：链路上实际跑了多少字节/秒

### NCCL

### gpu-burn

```
# https://github.com/wilicc/gpu-burn.git
/gpu_burn -m 90% 60        # 不开 TC
./gpu_burn -tc -m 90% 60    # 开 TC
```

- `gpu_burn gemm 5400 16384`
    - pgrep -af gpu_burn
    - gemm 分支显式使用 BF16 输入、FP32 累加，并调用 H100 Tensor Core。
        - $C_{ij}^{\mathrm{FP32}}=\sum_k A_{ik}^{\mathrm{BF16}}B_{kj}^{\mathrm{BF16}}$

### CPU

#### NUMA 与亲和性

```
> lscpu | grep -E "^(Model name|Socket|Core|Thread|NUMA node[01]? )"

Model name:                      Intel(R) Xeon(R) Platinum 8468
Thread(s) per core:              2
Core(s) per socket:              48
Socket(s):                       2
NUMA node0 CPU(s):               0-47,96-143
NUMA node1 CPU(s):               48-95,144-191
```

| NUMA节点 | CPU插槽 | 第一组硬件线程 | 第二组超线程 | 合计 |
|---|---|---|---|---:|
| NUMA 0 | Socket 0 | CPU 0–47 | CPU 96–143 | 96个逻辑CPU |
| NUMA 1 | Socket 1 | CPU 48–95 | CPU 144–191 | 96个逻辑CPU |

```
CPU 0  和 CPU 96   是同一个物理核心的两个超线程
CPU 1  和 CPU 97   是同一个物理核心的两个超线程
...
CPU 47 和 CPU 143  是同一个物理核心的两个超线程

CPU 48 和 CPU 144  是同一个物理核心的两个超线程
...
CPU 95 和 CPU 191  是同一个物理核心的两个超线程


cat /sys/devices/system/cpu/cpu0/topology/thread_siblings_list
cat /sys/devices/system/cpu/cpu48/topology/thread_siblings_list

0,96
48,144
```